# Meridional Overturning Circulation (MOC) Tutorial

This notebook demonstrates how to compute the Meridional Overturning Circulation (MOC) from FESOM2 vertical velocity data using the `fesomp.diag` module.

**Topics covered:**
- Computing global MOC
- Basin-specific MOC (Atlantic, Indo-Pacific)
- AMOC index calculation
- Using predefined basin masks

In [ ]:
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import fesomp

%matplotlib inline

## 1. Load Mesh and Vertical Velocity Data

In FESOM2, vertical velocity (`w`) is defined on **nodes** at **level interfaces** (nz levels).

In [ ]:
# Load mesh
mesh = fesomp.load_mesh('/Users/nkolduno/PYTHON/DATA/CORE27_mesh/fesom.mesh.diag.nc')
print(mesh)
print(f"\nMesh has {mesh.nlev} vertical levels (level interfaces)")
print(f"Nodes: {mesh.n2d}")

In [ ]:
# Load vertical velocity data
w_ds = xr.open_dataset('/Users/nkolduno/PYTHON/DATA/CORE27_data/w.fesom.1958.nc')
print(w_ds)

In [ ]:
# Extract vertical velocity
w = w_ds['w']  # shape: (time, nz, n2d)
print(f"Vertical velocity shape: {w.shape}")
print(f"Dimensions: {w.dims}")
print(f"\nw is on {w.shape[1]} level interfaces (nz)")
print(f"w is on {w.shape[2]} nodes (n2d)")

In [ ]:
# Get node areas (surface areas for simplicity)
node_area = mesh.geometry.node_area[0]
print(f"Node area shape: {node_area.shape}")

## 2. Compute Global MOC

The MOC streamfunction represents the zonally-integrated volume transport as a function of latitude and depth.

In [ ]:
# Compute global MOC for first time step
w_t0 = w[0].values  # (nz, n2d)
print(f"w shape for first timestep: {w_t0.shape}")

moc_global, lat_bins = fesomp.diag.moc(
    w_t0,
    node_area,
    mesh.lat,
    mesh.depth_levels,
    lat_range=(-80, 80),
    nlats=161,  # 1° resolution
)

print(f"MOC shape: {moc_global.shape}")
print(f"Latitude bins: {len(lat_bins)} ({lat_bins[0]:.0f}° to {lat_bins[-1]:.0f}°)")

In [ ]:
# Plot Global MOC
fig, ax = plt.subplots(figsize=(14, 6))

# Create meshgrid for plotting
LAT, DEPTH = np.meshgrid(lat_bins, mesh.depth_levels)

# Plot MOC
levels = np.linspace(-30, 30, 31)
cf = ax.contourf(LAT, DEPTH, moc_global, levels=levels, cmap='RdBu_r', extend='both')
ax.contour(LAT, DEPTH, moc_global, levels=levels[::2], colors='k', linewidths=0.5)

ax.set_ylim(5000, 0)  # Invert y-axis
ax.set_xlabel('Latitude (°N)')
ax.set_ylabel('Depth (m)')
ax.set_title('Global Meridional Overturning Circulation')
plt.colorbar(cf, ax=ax, label='Streamfunction (Sv)')
plt.tight_layout()
plt.show()

## 3. Atlantic MOC (AMOC)

To compute basin-specific MOC, we need to apply a basin mask. The `fesomp.diag` module includes predefined basin masks from GeoJSON files.

**Note:** Using basin masks requires `shapely` to be installed: `pip install shapely`

In [ ]:
# List available MOC basins
print("Available MOC basins:")
for basin in fesomp.diag.list_moc_basins():
    print(f"  - {basin}")

In [ ]:
# Get Atlantic basin mask
# This requires shapely to be installed
try:
    atl_mask = fesomp.diag.get_basin_mask(mesh.lon, mesh.lat, "Atlantic_MOC")
    print(f"Atlantic mask: {atl_mask.sum()} nodes out of {mesh.n2d} ({100*atl_mask.sum()/mesh.n2d:.1f}%)")
    has_shapely = True
except ImportError:
    print("shapely not installed - using simple box mask for Atlantic")
    # Simple box approximation for Atlantic
    atl_mask = (
        ((mesh.lon >= -80) & (mesh.lon <= 0) & (mesh.lat >= -35) & (mesh.lat <= 65)) |
        ((mesh.lon >= -80) & (mesh.lon <= 20) & (mesh.lat >= -60) & (mesh.lat < -35))
    )
    print(f"Simple Atlantic mask: {atl_mask.sum()} nodes")
    has_shapely = False

In [ ]:
# Visualize the Atlantic mask
fig, ax, _ = fesomp.plot(
    atl_mask.astype(float),
    mesh.lon,
    mesh.lat,
    mapproj="robin",
    titles="Atlantic Basin Mask",
    cmap="Blues",
    levels=(0, 1, 2),
    coastlines=True,
)
plt.show()

In [ ]:
# Compute Atlantic MOC
amoc, lat_bins = fesomp.diag.moc(
    w_t0,
    node_area,
    mesh.lat,
    mesh.depth_levels,
    lat_range=(-35, 70),
    nlats=106,
    mask=atl_mask,
)

print(f"AMOC shape: {amoc.shape}")

In [ ]:
# Plot Atlantic MOC
fig, ax = plt.subplots(figsize=(12, 6))

LAT, DEPTH = np.meshgrid(lat_bins, mesh.depth_levels)

levels = np.linspace(-20, 20, 21)
cf = ax.contourf(LAT, DEPTH, amoc, levels=levels, cmap='RdBu_r', extend='both')
ax.contour(LAT, DEPTH, amoc, levels=levels[::2], colors='k', linewidths=0.5)

# Mark RAPID array latitude
ax.axvline(x=26.5, color='green', linestyle='--', linewidth=2, label='RAPID (26.5°N)')

ax.set_ylim(5000, 0)
ax.set_xlabel('Latitude (°N)')
ax.set_ylabel('Depth (m)')
ax.set_title('Atlantic Meridional Overturning Circulation (AMOC)')
ax.legend(loc='lower right')
plt.colorbar(cf, ax=ax, label='Streamfunction (Sv)')
plt.tight_layout()
plt.show()

## 4. AMOC Index

The AMOC index is typically defined as the maximum overturning streamfunction at 26.5°N (RAPID array location) in the depth range 500-2000m.

In [ ]:
# Compute AMOC index
amoc_index = fesomp.diag.amoc_index(
    amoc,
    lat_bins,
    mesh.depth_levels,
    lat=26.5,
    depth_range=(500, 2000)
)

print(f"AMOC index at 26.5°N: {amoc_index:.2f} Sv")

In [ ]:
# Plot AMOC profile at 26.5°N
lat_idx = np.argmin(np.abs(lat_bins - 26.5))
amoc_profile = amoc[:, lat_idx]

fig, ax = plt.subplots(figsize=(6, 8))
ax.plot(amoc_profile, mesh.depth_levels, 'b-', linewidth=2)
ax.axvline(x=0, color='k', linestyle='--', linewidth=0.5)
ax.axhline(y=500, color='gray', linestyle=':', label='500m')
ax.axhline(y=2000, color='gray', linestyle=':', label='2000m')

# Mark maximum
depth_mask = (mesh.depth_levels >= 500) & (mesh.depth_levels <= 2000)
max_idx = np.argmax(amoc_profile[depth_mask])
max_depth = mesh.depth_levels[depth_mask][max_idx]
ax.plot(amoc_index, max_depth, 'ro', markersize=10, label=f'Max: {amoc_index:.1f} Sv')

ax.set_ylim(5000, 0)
ax.set_xlabel('Streamfunction (Sv)')
ax.set_ylabel('Depth (m)')
ax.set_title(f'AMOC Profile at {lat_bins[lat_idx]:.1f}°N')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Time Series of AMOC

Now let's compute the AMOC for all time steps to see the temporal evolution.

In [ ]:
# Compute AMOC for all time steps
w_all = w.values  # (time, nz, n2d)
print(f"Computing AMOC for {w_all.shape[0]} time steps...")

amoc_all, lat_bins = fesomp.diag.moc(
    w_all,
    node_area,
    mesh.lat,
    mesh.depth_levels,
    lat_range=(-35, 70),
    nlats=106,
    mask=atl_mask,
)

print(f"AMOC time series shape: {amoc_all.shape}")

In [ ]:
# Compute AMOC index time series
amoc_index_ts = fesomp.diag.amoc_index(
    amoc_all,
    lat_bins,
    mesh.depth_levels,
    lat=26.5,
    depth_range=(500, 2000)
)

print(f"AMOC index time series shape: {amoc_index_ts.shape}")
print(f"AMOC range: {amoc_index_ts.min():.2f} - {amoc_index_ts.max():.2f} Sv")
print(f"AMOC mean: {amoc_index_ts.mean():.2f} Sv")

In [ ]:
# Plot AMOC index time series
time_steps = np.arange(len(amoc_index_ts))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(time_steps, amoc_index_ts, 'b-o', linewidth=2, markersize=6)
ax.axhline(y=amoc_index_ts.mean(), color='r', linestyle='--', label=f'Mean: {amoc_index_ts.mean():.1f} Sv')

ax.set_xlabel('Time step')
ax.set_ylabel('AMOC Index (Sv)')
ax.set_title('AMOC Index at 26.5°N (500-2000m maximum)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Compare Global vs Atlantic MOC

In [ ]:
# Compute global MOC for comparison
moc_global_all, lat_global = fesomp.diag.moc(
    w_t0,
    node_area,
    mesh.lat,
    mesh.depth_levels,
    lat_range=(-80, 80),
    nlats=161,
)

# Plot side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Global MOC
ax = axes[0]
LAT, DEPTH = np.meshgrid(lat_global, mesh.depth_levels)
levels = np.linspace(-30, 30, 31)
cf = ax.contourf(LAT, DEPTH, moc_global_all, levels=levels, cmap='RdBu_r', extend='both')
ax.contour(LAT, DEPTH, moc_global_all, levels=levels[::2], colors='k', linewidths=0.3)
ax.set_ylim(5000, 0)
ax.set_xlabel('Latitude (°N)')
ax.set_ylabel('Depth (m)')
ax.set_title('Global MOC')
plt.colorbar(cf, ax=ax, label='Sv')

# Atlantic MOC
ax = axes[1]
LAT, DEPTH = np.meshgrid(lat_bins, mesh.depth_levels)
levels = np.linspace(-20, 20, 21)
cf = ax.contourf(LAT, DEPTH, amoc, levels=levels, cmap='RdBu_r', extend='both')
ax.contour(LAT, DEPTH, amoc, levels=levels[::2], colors='k', linewidths=0.3)
ax.set_ylim(5000, 0)
ax.set_xlabel('Latitude (°N)')
ax.set_ylabel('Depth (m)')
ax.set_title('Atlantic MOC')
plt.colorbar(cf, ax=ax, label='Sv')

plt.tight_layout()
plt.show()

## 7. Indo-Pacific MOC (Optional)

We can also compute the Indo-Pacific MOC using the corresponding mask.

In [ ]:
# Get Indo-Pacific mask
try:
    indopac_mask = fesomp.diag.get_basin_mask(mesh.lon, mesh.lat, "IndoPacific_MOC")
    print(f"Indo-Pacific mask: {indopac_mask.sum()} nodes ({100*indopac_mask.sum()/mesh.n2d:.1f}%)")
    
    # Compute Indo-Pacific MOC
    ipmoc, lat_ip = fesomp.diag.moc(
        w_t0,
        node_area,
        mesh.lat,
        mesh.depth_levels,
        lat_range=(-70, 65),
        nlats=136,
        mask=indopac_mask,
    )
    
    # Plot
    fig, ax = plt.subplots(figsize=(12, 6))
    LAT, DEPTH = np.meshgrid(lat_ip, mesh.depth_levels)
    levels = np.linspace(-30, 30, 31)
    cf = ax.contourf(LAT, DEPTH, ipmoc, levels=levels, cmap='RdBu_r', extend='both')
    ax.contour(LAT, DEPTH, ipmoc, levels=levels[::2], colors='k', linewidths=0.3)
    ax.set_ylim(5000, 0)
    ax.set_xlabel('Latitude (°N)')
    ax.set_ylabel('Depth (m)')
    ax.set_title('Indo-Pacific MOC')
    plt.colorbar(cf, ax=ax, label='Sv')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print("shapely not installed - skipping Indo-Pacific MOC")

## 8. Summary

### MOC Functions

- `fesomp.diag.moc(w, node_area, lat, depth_levels, ...)` - Compute MOC streamfunction
- `fesomp.diag.amoc_index(moc_data, lat_bins, depth_levels, ...)` - Compute AMOC index
- `fesomp.diag.get_basin_mask(lon, lat, basin_name)` - Get predefined basin mask
- `fesomp.diag.list_moc_basins()` - List available basin names

### Key Points

1. **FESOM2 w data**: Vertical velocity is on **nodes** at **level interfaces** (nz levels)
2. **Basin masks**: Require `shapely` for GeoJSON-based masks, or use simple box masks
3. **AMOC index**: Typically maximum at 26.5°N in 500-2000m depth range
4. **Units**: MOC is in Sverdrups (1 Sv = 10⁶ m³/s)

In [ ]:
# Clean up
plt.close('all')
print("\nMOC tutorial complete!")